In [1]:
import ollama
response = ollama.chat(model="llama3.1:8b",
                       messages=[{"role": "user", "content": "Say hello in one sentence."}])
print(response["message"]["content"])

Hello! How can I assist you today?


In [2]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("lufthansa")     # reopen the existing store

print("Loaded collection with", collection.count(), "docs")

Loaded collection with 185 docs


In [3]:
question = "What are the biggest risks for Lufthansa right now?"

# retrieve the 5 most relevant docs (semantic search)
results   = collection.query(query_texts=[question], n_results=5)
retrieved = results["documents"][0]
metas     = results["metadatas"][0]

# build ONE context string from the retrieved docs (tag each with its source)
context = "\n\n".join(f"[{m['source']}] {doc}" for doc, m in zip(retrieved, metas))

print(context)

[reddit] Lufthansa Reviews. We cannot provide a description for this page right now

[reddit] My (great) experience flying with Lufthansa. I don't often fly Lufthansa but eww what an experience.

[reddit] r/Lufthansa on Reddit: How your first time experience with Lufthansa.

[reddit] r/travel on Reddit: Flew Lufthansa for the first time (multiple flights), will try to avoid them in the future. Here are my impressions.

[news] Lufthansa Group Reports First Quarter 2026 Net Loss of €665 Million or €0.55 per Share. May 6, 2026 - Lufthansa Group has reported a first quarter net loss of €665 million or (€0.55) per share on a year-over-year increase in revenue of 8.0 percent to €8.7 billion. At March 31, 2026, the company had total available liquidity of €10.3 billion.


testing ollama with correct system_prompt and user_prompt

In [4]:
import ollama

system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided to answer — do not invent facts.
Be concise, specific, and ground every claim in the evidence."""

user_prompt = f"""Evidence:
{context}

Question: {question}

Answer as a strategic advisor, citing the evidence."""

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
)

print(response["message"]["content"])

Based on the provided evidence, I would identify two key risks for Lufthansa:

1. **Revenue Volatility**: The news article indicates that Lufthansa reported an 8.0% year-over-year increase in revenue, but this growth did not translate into profitability, as they still recorded a net loss of €665 million in the first quarter of 2026. This suggests that revenue growth may be inconsistent and not necessarily tied to profitability.
2. **Customer Dissatisfaction**: The Lufthansa Reviews thread on Reddit and multiple posts from travelers sharing negative experiences with Lufthansa (e.g., "eww what an experience" and "will try to avoid them in the future") suggest that there may be underlying issues with customer satisfaction, which could impact loyalty and repeat business. This is a concern for any airline, as customer retention is critical to long-term success.

These two risks should be monitored closely by Lufthansa's management team to ensure that they are addressed promptly and effectiv

In [5]:
import json, ollama

system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "supporting_evidence": list of 2-3 short evidence points taken from the context
- "expected_impact": the expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""

user_prompt = f"""Evidence:
{context}

Question: {question}"""

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ],
    format="json"          
)

rec = json.loads(response["message"]["content"])   
print(json.dumps(rec, indent=2))                   

{
  "recommendation": "Invest in improving customer experience",
  "supporting_evidence": [
    "Flew Lufthansa for the first time (multiple flights), will try to avoid them in the future. Here are my impressions.",
    "I don't often fly Lufthansa but eww what an experience.",
    "Lufthansa Group Reports First Quarter 2026 Net Loss of \u20ac665 Million"
  ],
  "expected_impact": "Improved customer satisfaction and potential increase in revenue",
  "risk_level": "High",
  "priority": "High"
}


In [6]:
PATH = "lufthansa_labeled.json"
labeled = json.load(open(PATH, encoding="utf-8"))

for d in labeled:
    if d["category"] == "opportunity":
        r = ollama.chat(model="llama3.1:8b", messages=[{"role": "user", "content":
            "Rate the business impact of this opportunity for Lufthansa. "
            "Answer with exactly ONE word — High, Medium, or Low.\n\n" + d["text"]}])
        ans = r["message"]["content"].strip().split()[0].strip(".,").capitalize()
        d["impact"] = ans if ans in ("High", "Medium", "Low") else "Medium"
        print(d["text"][:55], "->", d["impact"])

json.dump(labeled, open(PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Saved impact for", sum(d["category"] == "opportunity" for d in labeled), "opportunities")

Where Lufthansa’s quadjets are flying this summer. As L -> Low
Success Booking Lufthansa via Lifemiles, here are few t -> Medium
Barclays Lufthansa Miles & More Increased Sign Up Bonus -> Medium
Financial results | AIR FRANCE KLM. 1st Quarter results -> Low
KLM Royal Dutch Airlines - Book flights online - KLM US -> Medium
Air France KLM targets high tech cargo with. - Air Carg -> Medium
Why EasyJet's Share Price Has Surged 21% in a Month — A -> Medium
Emirates boosts long-haul fleet with mega order - PaxEx -> Medium
Turkish Airlines - airBaltic Expansion. Turkish Airline -> Low
Award-winning Lufthansa Allegris cabin now bookable for -> Medium
Media Library - newsroom.lufthansagroup.com. Lufthansa  -> Low
Investor Relations - Lufthansa Group Investor Relations -> Low
Saved impact for 12 opportunities


In [7]:
def ceo_agent(question, k=5):
    # 1. RETRIEVE evidence
    results   = collection.query(query_texts=[question], n_results=k)
    retrieved = results["documents"][0]
    metas     = results["metadatas"][0]
    context   = "\n\n".join(f"[{m['source']}] {doc}" for doc, m in zip(retrieved, metas))

    # 2. PROMPT
    system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "justification": 1-2 sentences explaining WHY this recommendation follows from the evidence
- "supporting_evidence": list of 2-3 short evidence points from the context
- "expected_impact": expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""
    user_prompt = f"Evidence:\n{context}\n\nQuestion: {question}"

    # 3. GENERATE (structured JSON)
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )

    # 4. PARSE + attach question and evidence URLs
    rec = json.loads(response["message"]["content"])
    rec["question"] = question
    rec["sources"]  = [m["url"] for m in metas]
    return rec

testing reusable code

In [8]:
import json
result = ceo_agent("What are the major opportunities for Lufthansa?")
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "recommendation": "Invest in improving customer experience to increase positive reviews and ratings",
  "justification": "The evidence suggests that while some customers have had negative experiences with Lufthansa, many others have reported positive ones. Improving the overall customer experience could lead to more positive reviews and higher ratings.",
  "supporting_evidence": [
    "Many customers have reported positive experiences flying with Lufthansa (reddit post)",
    "There are alternative airlines that are rated higher than Lufthansa, indicating room for improvement (RatingFacts article)",
    "Lufthansa is planning to retire a large portion of its long-haul fleet and shutter CityLine, which may indicate a need for change (The Explorer Blog article)"
  ],
  "expected_impact": "Increased customer loyalty, retention, and positive word-of-mouth",
  "risk_level": "Medium",
  "priority": "High",
  "question": "What are the major opportunities for Lufthansa?",
  "sources": [
  

In [9]:
questions = [
    "What are the major opportunities for Lufthansa?",
    "What are the biggest risks for Lufthansa?",
    "What are competitors doing?",
    "Which technologies or trends should Lufthansa management monitor?",
    "What strategic actions should Lufthansa prioritize?",
]

recommendations = []
for q in questions:
    print("Generating:", q)
    recommendations.append(ceo_agent(q))

json.dump(recommendations,
          open("recommendations.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\n Saved", len(recommendations), "recommendations")

Generating: What are the major opportunities for Lufthansa?
Generating: What are the biggest risks for Lufthansa?
Generating: What are competitors doing?
Generating: Which technologies or trends should Lufthansa management monitor?
Generating: What strategic actions should Lufthansa prioritize?

 Saved 5 recommendations


CEO Briefing (Section 7)

In [10]:
def ceo_briefing(recommendations):
    # summarize the recommendations as input
    rec_summary = "\n".join(
        f"- {r['recommendation']} (priority {r['priority']}, risk {r['risk_level']})"
        for r in recommendations
    )

    system_prompt = """You are chief of staff to the CEO of Lufthansa.
Write a concise executive briefing as a JSON object with EXACTLY these 3 keys:
- "what_happened": a single plain-text string (2-3 sentences)
- "why_it_matters": a single plain-text string (2-3 sentences)
- "what_to_do_next": a single plain-text string (2-3 sentences)
Each value MUST be a plain string — NOT a nested object, dict, or list.
Base it ONLY on the recommendations provided. Do not invent facts."""

    user_prompt = f"Strategic recommendations:\n{rec_summary}\n\nWrite the CEO briefing."

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )
    return json.loads(response["message"]["content"])

In [11]:
briefing = ceo_briefing(recommendations)
json.dump(briefing, open("ceo_briefing.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(json.dumps(briefing, indent=2, ensure_ascii=False))

{
  "what_happened": "Our team has identified several strategic areas that require immediate attention to drive growth and improve competitiveness.",
  "why_it_matters": "Investing in customer experience, addressing passenger satisfaction concerns, analyzing the competitive landscape, monitoring fuel-efficient technology, and integrating air freight activities are all crucial to maintaining our market share and staying ahead of competitors.",
  "what_to_do_next": "We recommend allocating resources immediately to address these areas with a focus on improving customer experience, conducting thorough analysis of passenger feedback, benchmarking key competitors, researching innovative aircraft technologies, and integrating logistics networks."
}


In [1]:
from retrieval import semantic_search, bm25_search, hybrid_search

d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
q = "What are the biggest risks for Lufthansa?"
print("SEMANTIC:", semantic_search(q, 3)[0]["text"][:70])
print("BM25    :", bm25_search(q, 3)[0]["text"][:70])
print("HYBRID  :", hybrid_search(q, 3)[0]["text"][:70])

SEMANTIC: Lufthansa Reviews. We cannot provide a description for this page right
BM25    : Best Alternatives to Lufthansa in 2026 | RatingFacts. Looking for alte
HYBRID  : Best Alternatives to Lufthansa in 2026 | RatingFacts. Looking for alte


In [3]:
from collections import defaultdict

def retrieve_evidence(query, k_each=5, final_k=5):
    """Run all 3 retrievers, pool results, dedup by URL, keep the best by consensus."""
    methods = {
        "semantic": semantic_search(query, k_each),
        "bm25":     bm25_search(query, k_each),
        "hybrid":   hybrid_search(query, k_each),
    }

    seen = {}                                   # url -> {doc, hits, rank_sum}
    for docs_list in methods.values():
        for rank, d in enumerate(docs_list):    # rank 0 = top of that method
            key = d["url"]
            if key not in seen:
                seen[key] = {"doc": d, "hits": 0, "rank_sum": 0}
            seen[key]["hits"]     += 1           # how many methods found it
            seen[key]["rank_sum"] += rank        # how high they ranked it

    # best = found by MOST methods, tie-break by best average rank
    ranked = sorted(seen.values(), key=lambda x: (-x["hits"], x["rank_sum"]))
    return [x["doc"] for x in ranked[:final_k]]

In [4]:
q = "What are the biggest risks for Lufthansa?"
best = retrieve_evidence(q)
for i, d in enumerate(best, 1):
    print(f"#{i}  {d['text'][:75]}")

#1  r/travel on Reddit: Flew Lufthansa for the first time (multiple flights), w
#2  Best Alternatives to Lufthansa in 2026 | RatingFacts. Looking for alternati
#3  My (great) experience flying with Lufthansa. I don't often fly Lufthansa bu
#4  Is Lufthansa worth it for a long haul flight? : r/Lufthansa. February 25, 2
#5  Lufthansa Reviews. We cannot provide a description for this page right now


In [6]:
for i, d in enumerate(retrieve_evidence("Lufthansa pilot strike fuel costs financial loss"), 1):
    print(f"#{i}  {d['text'][:1750]}")

#1  Lufthansa Group still sees improved profit despite fuel hit - FlightGlobal. May 6, 2026 - It comes after the group posted . the same period last year. Lufthansa Group’s net loss was cut by one-quarter, to €665 million, over the same period.
#2  Lufthansa Group Reports First Quarter 2026 Net Loss of €665 Million or €0.55 per Share. May 6, 2026 - Lufthansa Group has reported a first quarter net loss of €665 million or (€0.55) per share on a year-over-year increase in revenue of 8.0 percent to €8.7 billion. At March 31, 2026, the company had total available liquidity of €10.3 billion.
#3  The strike crisis at Lufthansa is escalating: Flights canceled,. News The strike crisis at Lufthansa is . The strike crisis at Lufthansa is escalating: Flights canceled, tens of thousands of passengers affected.
#4  Lufthansa Strike: 60% of Flights Axed Amid Pay Dispute. A 48-hour strike by Lufthansa pilots has grounded approximately 60% of the airline s flights. . Lufthansa pilots and those from its